In [81]:
from src import IKEAQueryGenerator
from src import VectorRetriever, RerankerManager, LLMHybridSummarization
from src import LLMQueryRewriter, SimplePromptConstructor
from src import BlackBoxQueryGenerator, WhiteBoxQueryLoader
from src import OpenAILLM
from src import RougeEvaluator, LiteralEvaluator, EmbeddingEvaluator, CrossEncoderEvaluator
import os
import json
import configs
from tqdm import tqdm
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
import argparse
from argparse import Namespace

def get_args(args_list=None):
    parser = argparse.ArgumentParser()
    # 基础输入
    parser.add_argument("--device", type=str, default="cuda:1")
    parser.add_argument("--cfg_name", type=str, default="fiqa", help="Config name in configs/")

    # retrieval
    parser.add_argument("--force_rebuild", action="store_true", help="Force rebuild retrieval database")

    # LLM
    parser.add_argument("--llm_model", type=str, default="./Models/Qwen2.5-7B-Instruct")
    parser.add_argument("--llm_base_url", type=str, default="http://localhost:22999/v1")
    parser.add_argument("--llm_api_key", type=str, default="EMPTY")
    parser.add_argument("--llm_temperature", type=float, default=0)
    parser.add_argument("--llm_top_p", type=float, default=1)
    parser.add_argument("--llm_max_gen_len", type=int, default=4096)

    # optional
    parser.add_argument("--reasoning", action="store_true", help="Whether to save the reasoning content of thinking models")
    parser.add_argument("--rewriter", action="store_true", help="Whether to use query rewriting")
    parser.add_argument("--reranker", action="store_true", help="Whether to use reranker")
    parser.add_argument("--summarizer", action="store_true", help="Whether to use summarization")

    # attack
    parser.add_argument("--attack", type=str, choices=["iega", "bbqg", "wbtq"], default="bbqg", help="Whether to use attack for query generation")
    parser.add_argument("--entity_file", type=str, default=None, help="Path to the entity file for better BBQG and iter attack")
    parser.add_argument("--attack_num", type=int, default=500, help="Number of attack queries to generate")
    parser.add_argument("--batch_size", type=int, default=50, help="Batch size for processing queries")
    
    if args_list is not None:
        return parser.parse_args(args_list)
    else:
        return parser.parse_args()  # 命令行模式

# 在 Jupyter 中使用
args = get_args(["--cfg_name", "fiqa", "--attack", "iega"])
# 或用默认值
# args = get_args([])

print(args)

Namespace(device='cuda:1', cfg_name='fiqa', force_rebuild=False, llm_model='./Models/Qwen2.5-7B-Instruct', llm_base_url='http://localhost:22999/v1', llm_api_key='EMPTY', llm_temperature=0, llm_top_p=1, llm_max_gen_len=4096, reasoning=False, rewriter=False, reranker=False, summarizer=False, attack='iega', entity_file=None, attack_num=500, batch_size=50)


In [10]:
def setup(cfg, args):
    # 初始化
    llm = OpenAILLM(model = args.llm_model, 
                    base_url = args.llm_base_url, 
                    api_key = args.llm_api_key, 
                    reasoning = args.reasoning,
                    temperature = args.llm_temperature,
                    top_p = args.llm_top_p,
                    max_gen_len = args.llm_max_gen_len,
                    max_workers=50)
    
    llm_tool = OpenAILLM(model = cfg.tool_llm["model"], 
                    base_url = cfg.tool_llm["base_url"], 
                    api_key = cfg.tool_llm["api_key"], 
                    reasoning = cfg.tool_llm["reasoning"],
                    temperature = cfg.tool_llm["temperature"],
                    top_p = cfg.tool_llm["top_p"],
                    max_workers = 50)

    query_rewriter = LLMQueryRewriter(llm_tool, cfg.data["description"])

    retriever = VectorRetriever(cfg, device=args.device)
    if cfg.reranker["model"]:
        reranker = RerankerManager(reranker_model=cfg.reranker["model"], top_n=cfg.retrieval['top_n'], device=args.device)
    else:
        reranker = None

    if args.rewriter and not args.reranker:
        print("[NOTING] Query rewriting is enabled but Reranker is disabled. It's recommended to use Query Rewriter with Reranker for better performance.")

    summarizer = LLMHybridSummarization(llm_tool, embed_provider=cfg.summarizer["provider"], embed_model_dir=cfg.summarizer["model"], device='cuda:1')
    constructor = SimplePromptConstructor()

    return llm, llm_tool, query_rewriter, retriever, reranker, summarizer, constructor

In [11]:
cfg = getattr(configs, args.cfg_name) if hasattr(configs, args.cfg_name) else None

In [12]:
llm, llm_tool, query_rewriter, retriever, reranker, summarizer, constructor = setup(cfg, args)

[INFO] Retrieval name: ./data/fiqa Store path: ./retrieval_stores/./data/fiqa/bge-large-en-v1.5/chroma
[INFO] Loading existing Chroma DB: ./data/fiqa
Retriever of mmr is ready.
Retriever of chroma is ready.
[INFO] Retriever for ./data/fiqa is ready!
[INFO] Reranker BAAI/bge-reranker-large is ready!
[INFO] Summarizer embedding model ./Models/BAAI-bge-large-en-v1.5 loaded successfully.


In [74]:
ikea = IKEAQueryGenerator(llm_tool, data_description=cfg.data["description"] ,device=args.device)

In [75]:
ikea._generate_new_words(number=100)

add entries (length:59) into full query DB...


In [77]:
ikea.shuffle_into_queries(prior_related_th=0.10, unsimilar_th=0.4)

筛选出27个与主题'Finance'相关且相似度低于0.4的条目, 当前可用query db长度27


In [78]:
print(ikea.full_query_db)
print(ikea.full_query_db_added_mask)

['Trust', 'Unemployment', 'Debt', 'Savings', 'Finance', 'Tax', 'VentureCapital', 'Budget', 'JunkBond', 'Valuation', 'StockExchange', 'Warrant', 'Portfolio', 'Equity', 'CapitalGains', 'Premium', 'Pension', 'Profit', 'MutualFund', 'Risk', 'Leasing', 'Shareholder', 'Loan', 'TreasuryBills', 'CreditRating', 'Insolvency', 'NetWorth', 'WealthManagement', 'Fiscal', 'Rental', 'Underwriting', 'HedgeFund', 'Merger', 'Recession', 'Futures', 'QuantitativeEasing', 'Subsidy', 'Inflation', 'Mortgage', 'ZeroCouponBond', 'Currency', 'InterestRate', 'Bond', 'Security', 'Revenue', 'Yield', 'Option', 'BalanceSheet', 'Economy', 'Liquidity', 'Investment', 'RealEstate', 'AnnualReport', 'Earnings', 'Property', 'Gold', 'Market', 'Dividend', 'Stock']
[False False False False False False  True  True False False False  True
  True False  True False False False  True False False False False  True
  True False  True False False  True False False  True  True  True  True
  True False False  True False  True  True  Tru

In [84]:
print(ikea.queries)

['VentureCapital', 'Budget', 'Warrant', 'Portfolio', 'CapitalGains', 'MutualFund', 'TreasuryBills', 'CreditRating', 'NetWorth', 'Rental', 'Merger', 'Recession', 'Futures', 'QuantitativeEasing', 'Subsidy', 'ZeroCouponBond', 'InterestRate', 'Bond', 'Security', 'Revenue', 'Yield', 'Option', 'BalanceSheet', 'AnnualReport', 'Property', 'Gold', 'Stock']


In [89]:
# --- experiment setting --- #
max_extraction_iteration = 768
if_debug = False
output_log_period = 50
generate_period = 1000

In [92]:
# --- extraction mode setting --- #
condition_match_mode = "softmax" # "random" or "greedy" or "soft_greedy" or "warm_up_greedy" or "softmax"


In [ ]:
# --- pipeline init --- #
count = 0 # 循环次数
new_anchor_word = None # 是否从变异得到了新锚点词
mutation_id = 0 # 变异ID
if condition_match_mode == "warm_up_greedy":
    current_mode = "random"
    print(f"Warmup start.\nInitialize mode: {current_mode}")
else:
    current_mode = condition_match_mode

In [ ]:
with tqdm(total=max_extraction_iteration) as pbar:
        while count < max_extraction_iteration:
            pass

In [90]:
if_generate_new = bool(count%generate_period==generate_period-1)

In [ ]:
if new_anchor_word is None: 
    # if no mutation, generate new anchor word
    anchor_word = ikea.query(
                        score_k=10,
                        condition_match_mode=current_mode, 
                        debug=(if_debug & bool(count % output_log_period==output_log_period-1)),
                        if_generate_new = if_generate_new,
                        max_retries= 3,
                        topic = topic,
                        generation_num = 100,
                        extra_demand= None,
                        shuffle_topic_th = 0.05,
                        shuffle_unsim_th = 0.7,
                        sample_temperature=sample_temperature
                        )
    is_mutation = False
else:                       
    # if has mutation, use the mutated word
    anchor_word = new_anchor_word